# Analisi delle dipendenze API YAML

**Author:** *Francesco Pezzuto*

## Glossario

- **Overview**: breve descrizione del contenuto del notebook
- **DataFrame**: struttura tabellare usata per analizzare i dati
- **Grafo**: insieme di nodi e archi che rappresentano relazioni tra componenti
- **Dipendenza**: relazione tra due elementi del sistema, ad esempio tra file o tra schemi
- **Peso della dipendenza**: numero di occorrenze con cui una dipendenza compare

# Overview

Il notebook analizza le dipendenze estratte da un insieme di specifiche OpenAPI in formato YAML.
L'obiettivo principale è quello di analizzare le dipendenze tra file, schemi, identificare componenti centrali per preparare i dati per la successiva fase di clustering e modularizzazione

### Origine dei dati

I dati del seguente file sono stati generati da una sequenza pipeline sviluppata in Kotlin che parsa i file YAML, estrae le ref, costruisce le dipendenze e calcola i pesi (numero di occorenze) di ogni dipendenza. Esporta i risultati in formato CSV

In [17]:
%use dataframe
import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.io.*

val dependencies = DataFrame.readCSV("C:\\Users\\Francesco.Pezzuto\\Desktop\\project\\demo\\src\\main\\kotlin\\com\\example\\demo\\output\\dependencies.csv")

val nodeStats = DataFrame.readCSV("C:\\Users\\Francesco.Pezzuto\\Desktop\\project\\demo\\src\\main\\kotlin\\com\\example\\demo\\output\\nodeStats.csv")


## Dataset: dependencies

Il dataset `dependencies` contiene le dipendenze estratte dalla pipeline.

Le colonne principali sono:
- `from`: nodo sorgente
- `to`: nodo destinazione
- `type`: tipo di dipendenza
- `count`: peso della dipendenza

Di seguito una breve illustrazione

In [18]:
dependencies

from,to,type,count
core.yaml,jsonapi.yaml,file_ref,387
items.yaml,jsonapi.yaml,file_ref,325
salesDoc.yaml,jsonapi.yaml,file_ref,194
surveyDoc.yaml,jsonapi.yaml,file_ref,126
purchaseDoc.yaml,jsonapi.yaml,file_ref,117
hierarchies.yaml,jsonapi.yaml,file_ref,107
customers.yaml,jsonapi.yaml,file_ref,102
tasks.yaml,jsonapi.yaml,file_ref,101
accessControl.yaml,jsonapi.yaml,file_ref,100
clusters.yaml,jsonapi.yaml,file_ref,95


## Dataset: nodeStats

Il dataset `nodeStats` contiene statistiche aggregate sui nodi del grafo delle dipendenze.

In particolare permette di osservare:
- grado in uscita
- grado in entrata
- centralità di base dei nodi

Di seguito una breve illustrazione

In [19]:
nodeStats

node,outDegree,inDegree,total
core.yaml,438,619,1057
jsonapi.yaml,2,3005,3007
items.yaml,426,63,489
salesDoc.yaml,279,33,312
surveyDoc.yaml,153,25,178
purchaseDoc.yaml,197,28,225
hierarchies.yaml,133,19,152
customers.yaml,172,28,200
tasks.yaml,116,22,138
accessControl.yaml,116,22,138


## Prime dipendenze più rilevanti

Di seguito vengono mostrate le dipendenze con peso maggiore.
Queste relazioni sono particolarmente interessanti perché indicano legami forti tra i componenti.

In [20]:
dependencies.sortByDesc("count").head(10);

from,to,type,count
core.yaml,jsonapi.yaml,file_ref,387
items.yaml,jsonapi.yaml,file_ref,325
salesDoc.yaml,jsonapi.yaml,file_ref,194
surveyDoc.yaml,jsonapi.yaml,file_ref,126
purchaseDoc.yaml,jsonapi.yaml,file_ref,117
hierarchies.yaml,jsonapi.yaml,file_ref,107
customers.yaml,jsonapi.yaml,file_ref,102
tasks.yaml,jsonapi.yaml,file_ref,101
accessControl.yaml,jsonapi.yaml,file_ref,100
clusters.yaml,jsonapi.yaml,file_ref,95


## Distribuzione delle dipendenze per tipo

Questa sezione mostra quanti archi appartengono a ciascuna categoria di dipendenza.

In [21]:
println("Dipendenze per tipo")
dependencies.groupBy("type").count()

Dipendenze per tipo


type,count
file_ref,228
external_schema_ref,1030
internal_schema_ref,1213
operation_external_schema_ref,1219
operation_internal_schema_ref,731


## Nodi più centrali

Questa tabella evidenzia i nodi con maggiore centralità, utile per identificare componenti condivisi o fortemente accoppiati.

In [22]:
nodeStats.sortByDesc("total").head(10)

node,outDegree,inDegree,total
jsonapi.yaml,2,3005,3007
core.yaml,438,619,1057
jsonapi.yaml::UNDEFINED_SCHEMA,0,993,993
jsonapi.yaml::failure,4,631,635
openapi.yaml,576,0,576
items.yaml,426,63,489
salesDoc.yaml,279,33,312
documents.yaml,132,160,292
purchaseDoc.yaml,197,28,225
customers.yaml,172,28,200


Durante il parsing, alcuni riferimenti non sono stati risolti fino al livello di schema specifico.
Per questi casi è stato introdotto il nodo `UNDEFINED_SCHEMA`, che rappresenta una dipendenza generica verso un file senza identificare il componente preciso.

Questo approccio permette di mantenere la completezza del grafo evitando la perdita di informazioni,
ma introduce nodi artificiali che verranno raffinati nelle fasi successive dell’analisi.

In [23]:
val complexFiles = dependencies
    .groupBy("from")
    .count()
    .sortByDesc("count")
    .head(10)

complexFiles


from,count
openapi.yaml,52
core.yaml::UserProfile,10
core.yaml,8
budgets.yaml::Budget,8
documents.yaml::DocHeaderAttributes,8
core.yaml::Organization,8
suppliers.yaml::Supplier,8
salesDoc.yaml,7
customers.yaml,7
items.yaml::ItemAttributes,7


In [24]:
val undefined = dependencies.filter {
    it["to"].toString().contains("UNDEFINED_SCHEMA")
}

undefined.head(20)
undefined.groupBy("from").count().sortByDesc("count")

from,count
purchaseDoc.yaml::GET /purchase-docs/...,3
salesDoc.yaml::GET /sales-docs/headers,3
salesDoc.yaml::GET /sales-docs/lines,3
warehouseDoc.yaml::GET /warehouse-doc...,3
purchaseDoc.yaml::GET /purchase-docs/...,3
purchaseDoc.yaml::POST /purchase-docs...,3
salesDoc.yaml::POST /sales-docs/heade...,3
salesDoc.yaml::POST /sales-docs/lines,3
items.yaml::GET /items/sheets,3
customers.yaml::GET /customer-account...,3


In [25]:
undefined.groupBy("type").count()

type,count
operation_external_schema_ref,440


In [26]:
val opExternal = dependencies.filter { it["type"] == "operation_external_schema_ref" }
opExternal.groupBy("to").count().sortByDesc("count").head(20)

to,count
jsonapi.yaml::failure,630
jsonapi.yaml::UNDEFINED_SCHEMA,292
core.yaml::UNDEFINED_SCHEMA,132
jsonapi.yaml::info,33
core.yaml::KPIEntityListResponse,14
core.yaml::KPIEntityResponse,12
documents.yaml::UNDEFINED_SCHEMA,10
entityAttributes.yaml::UNDEFINED_SCHEMA,6
core.yaml::Tags,5
entityAttributes.yaml::EntitySheetLis...,5


In [27]:
opExternal.groupBy("from").count().sortByDesc("count").head(20)

from,count
purchaseDoc.yaml::POST /purchase-docs...,5
salesDoc.yaml::POST /sales-docs/heade...,5
salesDoc.yaml::POST /sales-docs/lines,5
customers.yaml::PATCH /customer-accou...,5
items.yaml::PATCH /items/tags,5
items.yaml::GET /items/sheets,5
customers.yaml::GET /customer-account...,5
items.yaml::GET /item-instances/sheets,5
purchaseDoc.yaml::GET /purchase-docs/...,4
warehouseDoc.yaml::GET /warehouse-doc...,4
